# Проект 2 — Разпознаване на медицинска аномалия (CAMELYON17) — Colab

Този notebook сваля **подмножество** от слайдове на CAMELYON, извлича патчове,
прави EDA и тренира моделите (CNN / VGG / U-Net) върху GPU.

Преизползва модулите от `projects/anomaly/` (`data`, `eda`, `models`, `train`,
`metrics`). Пускай клетките отгоре надолу.

> **Важно:** пълният CAMELYON е ~2.95 TB. Тук сваляме само няколко слайда
> (tumor с анотации + normal), извличаме балансиран набор патчове и го кешираме
> в Google Drive, за да не сваляме всеки път.

## 1. Среда: GPU, системни пакети, Python зависимости

In [ ]:
!nvidia-smi -L  # провери че има GPU (Runtime -> Change runtime type -> GPU)

In [ ]:
# openslide е нативна библиотека за whole-slide .tif изображения
!apt-get -qq install -y openslide-tools > /dev/null
!pip -q install openslide-python awscli torchmetrics streamlit

## 2. Кодът на проекта

Целта е `import projects.anomaly` да работи. Избери **един** от двата варианта:

* **Вариант A — GitHub (препоръчан за промени по кода):** сложи `REPO_URL`.
  При нова версия само пускаш отново тази клетка — прави `git pull`.
* **Вариант Б — Google Drive (резервен):** остави `REPO_URL = ''`, качи папката
  в Drive и коригирай `DRIVE_DIR`.

In [ ]:
import os, sys

# ---- Вариант A: GitHub (по подразбиране) ----
REPO_URL = 'https://github.com/plamena-ilieva/camelyon-anomaly.git'
REPO_DIR = '/content/camelyon-anomaly'

# ---- Вариант Б: Google Drive (ако REPO_URL = '') ----
DRIVE_DIR = '/content/drive/MyDrive/camelyon-anomaly'  # коригирай при нужда

if REPO_URL:
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull            # вече клонирано -> тегли най-новото
    else:
        !git clone {REPO_URL} {REPO_DIR}
else:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = DRIVE_DIR

assert os.path.isdir(os.path.join(REPO_DIR, 'projects', 'anomaly')), \
    f'Не намирам projects/anomaly в {REPO_DIR}'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Изчисти кешираните модули, за да влязат промените след git pull
# (иначе Python пази старата версия в паметта и липсват новите функции)
for _m in [m for m in list(sys.modules) if m.split('.')[0] == 'projects']:
    del sys.modules[_m]

from projects.anomaly import data, eda, models, train, metrics
print('OK — модулите се импортират от', REPO_DIR)
print('има extract_tiles_balanced:', hasattr(data, 'extract_tiles_balanced'))

## 3. Разглеждане на S3 bucket-а на CAMELYON

Данните са в публичния bucket `s3://camelyon-dataset` (регион `us-west-2`),
достъпен без AWS акаунт с `--no-sign-request`. Първо разгледай структурата и
избери реални пътища (имената по-долу са типични, но ги потвърди с листването).

In [ ]:
# Най-горно ниво (очаквай нещо като CAMELYON16/ и CAMELYON17/)
!aws s3 ls --no-sign-request s3://camelyon-dataset/

In [ ]:
# Реалната структура: CAMELYON17/images/*.tif, /annotations/*.xml, /masks/*.tif
!aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON17/
print('--- примерни изображения и анотации ---')
!aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON17/images/ | head -5
!aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON17/annotations/ | head -5

## 4. Сваляне на малко подмножество слайдове

Свали 1–2 tumor слайда (с техните XML анотации) и 1–2 normal слайда. Коригирай
пътищата спрямо реалното листване от стъпка 3.

In [ ]:
import os, subprocess

RAW_DIR = '/content/camelyon_raw'
os.makedirs(RAW_DIR, exist_ok=True)
BUCKET = 's3://camelyon-dataset'
PREFIX = 'CAMELYON17'
N_TUMOR, N_NORMAL = 2, 2   # внимание: всеки слайд е ~2-3 GB; започни с 1, 1

def s3_ls(prefix):
    out = subprocess.run(['aws', 's3', 'ls', '--no-sign-request', f'{BUCKET}/{prefix}'],
                         capture_output=True, text=True).stdout
    return [ln.split()[-1] for ln in out.splitlines() if ln.split()]

imgs = sorted(f for f in s3_ls(f'{PREFIX}/images/') if f.endswith('.tif'))
ann = sorted(f for f in s3_ls(f'{PREFIX}/annotations/') if f.endswith('.xml'))
masks = sorted(f for f in s3_ls(f'{PREFIX}/masks/') if f.endswith('.tif'))

ann_base = {os.path.splitext(a)[0] for a in ann}
mask_base = {m[:-len('_mask.tif')] for m in masks}

# tumor = слайд с XML анотация (нужна за етикетиране на патчовете)
tumor_slides = [im for im in imgs if os.path.splitext(im)[0] in ann_base][:N_TUMOR]
# normal = слайд без анотация И без маска (със сигурност без тумор)
normal_slides = [im for im in imgs
                 if os.path.splitext(im)[0] not in ann_base
                 and os.path.splitext(im)[0] not in mask_base][:N_NORMAL]
print('tumor :', tumor_slides)
print('normal:', normal_slides)

def cp(src, dst):
    print('  сваляне', src.split('/')[-1])
    subprocess.run(['aws', 's3', 'cp', '--no-sign-request', src, dst], check=True)

TUMOR_SLIDES, NORMAL_SLIDES = [], []
for im in tumor_slides:
    cp(f'{BUCKET}/{PREFIX}/images/{im}', f'{RAW_DIR}/{im}')
    xml = os.path.splitext(im)[0] + '.xml'
    cp(f'{BUCKET}/{PREFIX}/annotations/{xml}', f'{RAW_DIR}/{xml}')
    TUMOR_SLIDES.append(im)
for im in normal_slides:
    cp(f'{BUCKET}/{PREFIX}/images/{im}', f'{RAW_DIR}/{im}')
    NORMAL_SLIDES.append(im)

print('Свалени:', sorted(os.listdir(RAW_DIR)))

## 5. Извличане на патчове (tiles)

Ползваме `data.extract_tiles_balanced`: за **tumor** слайд семплираме патчове с
център **вътре в анотираните полигони** (гарантирано tumor), а **normal**
патчовете идват от normal слайдовете. Така избягваме проблема при пълно
сканиране, при което лимитът се изчерпва върху нормална тъкан и не стига до
малките лезии (затова преди излизаха само normal патчове).

`LEVEL` контролира резолюцията (по-голям = по-бързо). Координатите на анотациите
са в ниво-0, кодът ги привежда автоматично.

In [ ]:
import time
import numpy as np

TILE_SIZE = 96
LEVEL = 2                  # 0 = пълна резолюция (бавно), 2 = по-бързо
N_TUMOR_PER_SLIDE = 1000   # tumor патчове от всеки tumor слайд
N_NORMAL_PER_SLIDE = 1000  # normal патчове от всеки normal слайд

all_patches, all_labels = [], []


def add(tiles):
    for patch, label in tiles:
        all_patches.append(patch)
        all_labels.append(label)


print(f'TUMOR слайдове: {TUMOR_SLIDES}')
print(f'NORMAL слайдове: {NORMAL_SLIDES}')
print('=' * 70)

# tumor слайдове: патчове с център ВЪТРЕ в полигоните (гарантирано tumor)
for i, name in enumerate(TUMOR_SLIDES, 1):
    print(f'[{i}/{len(TUMOR_SLIDES)}] TUMOR слайд: {name}')
    xml = os.path.join(RAW_DIR, os.path.splitext(name)[0] + '.xml')
    t0 = time.time()
    tiles = data.extract_tiles_balanced(
        os.path.join(RAW_DIR, name), tile_size=TILE_SIZE, level=LEVEL,
        annotation_path=xml, n_tumor=N_TUMOR_PER_SLIDE, n_normal=0, verbose=True)
    add(tiles)
    print(f'  -> {len(tiles)} патча за {time.time() - t0:.1f}s | тотал досега: {len(all_patches)}')
    print('-' * 70)

# normal слайдове: само normal патчове
for i, name in enumerate(NORMAL_SLIDES, 1):
    print(f'[{i}/{len(NORMAL_SLIDES)}] NORMAL слайд: {name}')
    t0 = time.time()
    tiles = data.extract_tiles_balanced(
        os.path.join(RAW_DIR, name), tile_size=TILE_SIZE, level=LEVEL,
        annotation_path=None, n_tumor=0, n_normal=N_NORMAL_PER_SLIDE, verbose=True)
    add(tiles)
    print(f'  -> {len(tiles)} патча за {time.time() - t0:.1f}s | тотал досега: {len(all_patches)}')
    print('-' * 70)

all_labels = np.array(all_labels)
print('=' * 70)
print('Общо патчове:', len(all_patches), '| по класове:', eda.class_distribution(all_labels))

In [ ]:
# Балансиране: равен брой tumor и normal
rng = np.random.default_rng(23)
tumor_idx = np.where(all_labels == data.TUMOR)[0]
normal_idx = np.where(all_labels == data.NORMAL)[0]
k = min(len(tumor_idx), len(normal_idx))
keep = np.concatenate([rng.choice(tumor_idx, k, replace=False),
                       rng.choice(normal_idx, k, replace=False)])
rng.shuffle(keep)

patches = [all_patches[i] for i in keep]
labels = all_labels[keep]
print('След балансиране:', eda.class_distribution(labels))

In [ ]:
# Кеш на патчовете -> Drive ако е монтиран, иначе локално в /content (ефимерно).
SAVE_DIR = '/content/drive/MyDrive' if os.path.isdir('/content/drive/MyDrive') else '/content'
CACHE = os.path.join(SAVE_DIR, 'camelyon_patches.npz')
np.savez_compressed(CACHE, patches=np.stack(patches), labels=labels)
print('Записано в', CACHE)
# По-късно: d = np.load(CACHE); patches, labels = list(d['patches']), d['labels']
# За да оцелее между сесии, монтирай Drive преди това:
#   from google.colab import drive; drive.mount('/content/drive')

## 6. EDA — разглеждане на данните (стъпка 2)

In [ ]:
import json
report = eda.summarize(patches, labels)
print(json.dumps(report, indent=2, ensure_ascii=False))

os.makedirs('figures', exist_ok=True)
eda.plot_class_distribution(labels, 'figures/class_distribution.png')
eda.plot_sample_grid(patches, list(labels), 'figures/sample_grid.png')

from IPython.display import Image as IPImage, display
display(IPImage('figures/class_distribution.png'), IPImage('figures/sample_grid.png'))

## 7. Подготовка на DataLoader-и

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split

full = data.PatchDataset(patches, labels, transform=data.default_transform(train=True))
n_val = max(1, int(0.2 * len(full)))
n_train = len(full) - n_val
train_ds, val_ds = random_split(full, [n_train, n_val],
                                generator=torch.Generator().manual_seed(23))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print('train:', n_train, 'val:', n_val, '| device:', train.get_device())

## 8. Стъпка 3 — CNN

In [ ]:
cnn = models.SimpleCNN(num_classes=2)
hist_cnn = train.fit(cnn, train_loader, val_loader, num_epochs=10, lr=1e-3)
print('CNN val:', train.evaluate(cnn, val_loader))

## 9. Стъпка 4 — VGG варианти

In [ ]:
results = {}
for cfg in ['VGG11', 'VGG16']:
    vgg = models.VGG(config=cfg, num_classes=2)
    train.fit(vgg, train_loader, val_loader, num_epochs=10, lr=1e-3)
    results[cfg] = train.evaluate(vgg, val_loader)
    print(cfg, results[cfg])

## 10. Стъпка 5 — U-Net автоенкодер (unsupervised anomaly detection)

Тренираме за реконструкция **само върху normal патчове**. После аномалията се
засича по грешката на реконструкцията: tumor патчовете би трябвало да имат
по-висок MSE, защото моделът не ги е виждал.

In [ ]:
from torch import nn

# DataLoader само с normal патчове, без нормализация (за чиста реконструкция)
normal_only = [p for p, l in zip(patches, labels) if l == data.NORMAL]
recon_tf = data.transforms.Compose([data.transforms.ToTensor()])
normal_ds = data.PatchDataset(normal_only, [0] * len(normal_only), transform=recon_tf)
normal_loader = DataLoader(normal_ds, batch_size=32, shuffle=True, num_workers=2)

unet = models.UNetAutoencoder(in_channels=3, out_channels=3).to(train.get_device())
opt = torch.optim.AdamW(unet.parameters(), lr=1e-3)
criterion = nn.MSELoss()
device = train.get_device()

for epoch in range(10):
    unet.train(); running = 0.0
    for images, _ in normal_loader:
        images = images.to(device)
        opt.zero_grad()
        loss = criterion(unet(images), images)
        loss.backward(); opt.step()
        running += loss.item()
    print(f'epoch {epoch}: recon MSE = {running / len(normal_loader):.4f}')

In [ ]:
# Разделя ли грешката на реконструкцията normal от tumor?
tumor_only = [p for p, l in zip(patches, labels) if l == data.TUMOR][:200]
norm_eval = [p for p, l in zip(patches, labels) if l == data.NORMAL][:200]

def err(plist):
    batch = torch.stack([recon_tf(p.astype('uint8')) for p in plist])
    return train.reconstruction_error(unet, batch)

e_tumor, e_normal = err(tumor_only), err(norm_eval)
print(f'recon MSE — normal: {e_normal.mean():.4f} | tumor: {e_tumor.mean():.4f}')

import matplotlib.pyplot as plt
plt.hist(e_normal.numpy(), bins=30, alpha=0.6, label='normal')
plt.hist(e_tumor.numpy(), bins=30, alpha=0.6, label='tumor')
plt.xlabel('reconstruction MSE (anomaly score)'); plt.legend()
plt.savefig('figures/anomaly_scores.png'); plt.show()

## 11. Запазване на модел за Streamlit (стъпка 6)

In [ ]:
# Запази най-добрия класификатор. В app.py избери същата архитектура.
SAVE_DIR = '/content/drive/MyDrive' if os.path.isdir('/content/drive/MyDrive') else '/content'
MODEL_PATH = os.path.join(SAVE_DIR, 'model.pt')
torch.save(cnn.state_dict(), MODEL_PATH)
print('Запазено:', MODEL_PATH, '— свали го и го посочи в Streamlit приложението.')